# Qwen2.5-VL-7B-Instruct — LoRA Fine-Tuning (Türkçe Belge OCR)

Bu notebook, `dataset_1000.zip` içindeki (kendi içinde zaten `train/valid/test` olarak bölünmüş) veriyle Qwen2.5-VL-7B-Instruct modeline **LoRA fine-tuning** uygular.

**Yaklaşım — neden LoRA:** 7B modelin tamamını (full fine-tune) eğitmek hem çok daha fazla VRAM hem çok daha fazla veri ister. LoRA, modelin ana ağırlıklarını dondurup (frozen) sadece küçük "adaptör" katmanları ekleyip onları eğitir — bu veri setiyle bile anlamlı bir adaptasyon sağlar, 80GB GPU'da rahat sığar.

**Adımlar:**
1. Drive bağla, kütüphaneleri kur
2. `dataset_1000.zip`'i Drive'dan **local diske** çıkar (Drive'dan tek tek okumak yavaş — bunu daha önce baseline testte de gördük)
3. `train/`, `valid/` ve `test/` klasörlerini oku (veri zaten önceden bölünmüş geliyor, tekrar bölmüyoruz)
4. Modeli eğitim için yükle, LoRA adaptörünü sadece **dil modeli katmanlarına** ekle (görsel encoder dondurulur)
5. Dataset + collate_fn tanımla (prompt kısmını maskeleyip sadece cevap üzerinden loss hesapla)
6. 80GB GPU'yu verimli kullanmak için batch size'ı deneyerek bul
7. Eğitimi başlat (`Trainer`)
8. LoRA adaptörünü Drive'a kaydet
9. Fine-tune sonrası, `dataset_1000/test/` klasöründeki **holdout örneklerde** CER ölç, grup bazlı özet çıkar

> `test/` eğitimde kullanılmaz; yalnız sonda CER için. Baseline aynı holdout. Gruplar: A temiz, B çok az bozulma, C belirgin, D bariz kötü. Örnekler: `QWEN_VL_ocr/data/`.

**Kaynak metin:** Hugging Face `erdem-erdem/Turkish-Law-Documents-700k-clustered` (Yargıtay + Danıştay kamuya açık karar metinleri).

**Rol:** Okuyucu LoRA. Görsel encoder donuk. Çıktı Drive `qwen_vl_7b_lora_finetuned/`. Klasör: `QWEN_VL_ocr/`.


## 1. Kurulum

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# transformers/accelerate: model + eğitim altyapısı
# peft: LoRA adaptörü için
# qwen-vl-utils: görsel ön işleme (process_vision_info)
# jiwer: CER hesabı (fine-tune sonrası karşılaştırma için)
!pip install -q -U transformers accelerate peft qwen-vl-utils jiwer

# Colab'da önceden yüklü gelen torchao (0.10.0), yeni peft sürümünün beklediği
# minimum sürümden (0.16.0) eski olduğu için get_peft_model() ImportError fırlatıyor.
# torchao bizim bf16 LoRA akışımızda hiç kullanılmıyor (quantization-aware training için
# opsiyonel bir bağımlılık) — en temiz çözüm onu tamamen kaldırmak.
!pip uninstall -y -q torchao

In [ ]:
import os
import json
import glob
import random
import shutil
import zipfile

import torch
from torch.utils.data import Dataset
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
from qwen_vl_utils import process_vision_info
import jiwer

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Toplam VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. Yollar ve `dataset_1000.zip`'i local diske çıkar

Baseline testte gördüğümüz gibi, Drive mount'u üzerinden çok sayıda küçük dosyayı tek tek okumak (network FUSE mount olduğu için) çok yavaş. Eğitim sırasında her epoch'ta 900+ görüntüyü tekrar tekrar okuyacağımız için, zip'i **bir kere** Drive'dan local diske (`/content/`) çıkarıp oradan çalışmak, tüm eğitim boyunca ciddi zaman kazandırır.

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/qwen_vl_7b_model"
DATASET_ZIP_PATH = "/content/drive/MyDrive/dataset_1000.zip"
LOCAL_DATASET_ROOT = "/content/dataset_1000"

LORA_OUTPUT_DIR_DRIVE = "/content/drive/MyDrive/qwen_vl_7b_lora_finetuned"
LOCAL_CHECKPOINT_DIR = "/content/lora_checkpoints"
RESULTS_OUTPUT_PATH_DRIVE = "/content/drive/MyDrive/finetuned_test_results.json"

assert os.path.isdir(MODEL_PATH), f"Model klasörü bulunamadı: {MODEL_PATH}"
assert os.path.isfile(DATASET_ZIP_PATH), f"dataset_1000.zip bulunamadı: {DATASET_ZIP_PATH}"

if not os.path.isdir(LOCAL_DATASET_ROOT):
    os.makedirs(LOCAL_DATASET_ROOT, exist_ok=True)
    print("dataset_1000.zip çıkartılıyor (bir kereye mahsus, birkaç dakika sürebilir)...")
    with zipfile.ZipFile(DATASET_ZIP_PATH, "r") as zf:
        zf.extractall(LOCAL_DATASET_ROOT)
    print("Bitti.")
else:
    print("Local dataset klasörü zaten mevcut, çıkartma atlandı.")

# Zip kendi içinde tek bir üst klasör oluşturmuş olabilir (örn. "dataset_1000/train/...").
# Böyle bir durumda gerçek train/valid/test klasörlerini bulmak için bir seviye içeri iniyoruz.
def _find_split_root(root: str) -> str:
    entries = [e for e in os.listdir(root) if not e.startswith(".")]
    has_splits = any(e.lower() in ("train", "valid", "val", "test") for e in entries)
    if has_splits:
        return root
    if len(entries) == 1 and os.path.isdir(os.path.join(root, entries[0])):
        return _find_split_root(os.path.join(root, entries[0]))
    return root

LOCAL_DATASET_ROOT = _find_split_root(LOCAL_DATASET_ROOT)
print(f"Dataset kök klasörü: {LOCAL_DATASET_ROOT}")
print("İçerik:", os.listdir(LOCAL_DATASET_ROOT))

## 3. Train/valid/test klasörlerini oku

`dataset_1000.zip` zaten `train/`, `valid/` (veya `val/`) ve `test/` alt klasörlerine bölünmüş halde geliyor — biz burada tekrar bölme yapmıyoruz, sadece her klasördeki mevcut `.png`/`.txt` çiftlerini listeliyoruz. Zip içindeki bu `test/` klasörü, fine-tune sonrası performansı ölçeceğimiz **asıl holdout set**.

In [ ]:
def _resolve_split_dir(root: str, names: list) -> str:
    for name in names:
        for candidate in (name, name.lower(), name.capitalize()):
            path = os.path.join(root, candidate)
            if os.path.isdir(path):
                return path
    raise FileNotFoundError(f"{names} isimlerinden hiçbiri {root} altında bulunamadı: {os.listdir(root)}")


TRAIN_DIR = _resolve_split_dir(LOCAL_DATASET_ROOT, ["train"])
VALID_DIR = _resolve_split_dir(LOCAL_DATASET_ROOT, ["valid", "val"])
TEST_DIR = _resolve_split_dir(LOCAL_DATASET_ROOT, ["test"])


def list_ids(split_dir: str) -> list:
    png_files = sorted(glob.glob(os.path.join(split_dir, "*.png")))
    return [os.path.splitext(os.path.basename(p))[0] for p in png_files]


train_ids = list_ids(TRAIN_DIR)
valid_ids = list_ids(VALID_DIR)
test_ids = list_ids(TEST_DIR)

print(f"TRAIN_DIR = {TRAIN_DIR} -> {len(train_ids)} örnek")
print(f"VALID_DIR = {VALID_DIR} -> {len(valid_ids)} örnek")
print(f"TEST_DIR  = {TEST_DIR} -> {len(test_ids)} örnek (fine-tune sonrası holdout değerlendirmesi bunun üzerinde yapılacak)")


def group_breakdown(split_dir: str, ids: list) -> dict:
    log_path = os.path.join(split_dir, "augmentation_log.json")
    log = {}
    if os.path.isfile(log_path):
        with open(log_path, "r", encoding="utf-8") as f:
            log = json.load(f)
    counts = {}
    for sid in ids:
        g = log.get(sid, {}).get("group", sid.split("_")[0])
        counts[g] = counts.get(g, 0) + 1
    return counts


print("\nTrain grup dağılımı:", group_breakdown(TRAIN_DIR, train_ids))
print("Valid grup dağılımı:", group_breakdown(VALID_DIR, valid_ids))
print("Test grup dağılımı: ", group_breakdown(TEST_DIR, test_ids))

## 4. Modeli eğitim için yükle

In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",  # PyTorch'un yerleşik memory-efficient attention'ı
)
model.to("cuda")

processor = AutoProcessor.from_pretrained(MODEL_PATH)
# Eğitimde generate() çağrılmıyor (teacher forcing), bu yüzden inference'taki gibi
# left-padding değil, standart right-padding kullanıyoruz.
processor.tokenizer.padding_side = "right"

print("Model ve processor yüklendi.")

## 5. LoRA adaptörünü kur

Görsel encoder'ı (`model.visual`) dondurup, LoRA'yı **sadece dil modelinin** attention/MLP projeksiyon katmanlarına ekliyoruz. Modelin katman isimlerini otomatik tarayıp `visual` alt modülü dışındaki `q_proj/k_proj/v_proj/o_proj/gate_proj/up_proj/down_proj` katmanlarını buluyoruz — bu, transformers sürümleri arasında tam modül yolu değişse bile sağlam çalışır.

In [ ]:
# Önce her şeyi dondur
for param in model.parameters():
    param.requires_grad = False

TARGET_SUFFIXES = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")

lora_target_modules = sorted({
    name for name, module in model.named_modules()
    if isinstance(module, torch.nn.Linear)
    and name.split(".")[-1] in TARGET_SUFFIXES
    and "visual" not in name
})
print(f"{len(lora_target_modules)} LoRA hedef katmanı bulundu. Örnekler:")
for n in lora_target_modules[:5]:
    print(" ", n)

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=lora_target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Gradient checkpointing: backward için tüm ara aktivasyonları bellekte tutmak yerine
# yeniden hesaplar. Belgeler binlerce görsel token ürettiği için bu, VRAM kullanımını
# ciddi oranda azaltır (biraz hız kaybı pahasına). PEFT ile birlikte kullanılırken
# enable_input_require_grads() gerekli, aksi halde gradyan akışı kesilir.
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False
print("Gradient checkpointing açıldı.")

## 6. Dataset ve collate_fn

Her örnek için tam konuşma (`user: [görsel + OCR talimatı]`, `assistant: [ground truth metin]`) oluşturulur. Loss'un sadece **modelin üretmesi gereken kısımda** (assistant cevabı) hesaplanması için, prompt kısmının token uzunluğunu ayrıca hesaplayıp `labels` içinde o kısmı `-100` (yoksay) ile maskeliyoruz. Padding token'ları da aynı şekilde maskelenir.

In [ ]:
OCR_PROMPT = (
    "Bu belgedeki metni eksiksiz ve doğru şekilde transkribe et. "
    "Sadece belgede yazan metni, satır satır ve orijinaline sadık kalarak yaz. "
    "Yorum, açıklama veya başlık ekleme; markdown biçimlendirmesi kullanma."
)


class QwenOCRDataset(Dataset):
    def __init__(self, sample_ids: list, dataset_dir: str):
        self.sample_ids = sample_ids
        self.dataset_dir = dataset_dir

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sid = self.sample_ids[idx]
        image_path = os.path.join(self.dataset_dir, f"{sid}.png")
        txt_path = os.path.join(self.dataset_dir, f"{sid}.txt")
        with open(txt_path, "r", encoding="utf-8") as f:
            target_text = f.read()
        return {"id": sid, "image_path": image_path, "target_text": target_text}


def build_user_message(image_path: str) -> list:
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": OCR_PROMPT},
        ],
    }]


def collate_fn(batch: list) -> dict:
    # 1) Tam konuşmayı (prompt + doğru cevap) toplu işleyip input_ids/pixel_values üret
    full_messages = [
        build_user_message(item["image_path"]) + [
            {"role": "assistant", "content": [{"type": "text", "text": item["target_text"]}]}
        ]
        for item in batch
    ]
    full_texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in full_messages
    ]
    image_inputs, _ = process_vision_info(full_messages)

    full_inputs = processor(
        text=full_texts,
        images=image_inputs,
        padding=True,
        return_tensors="pt",
    )

    # 2) Her örnek için prompt uzunluğunu (sadece user kısmı) tek tek, padding olmadan hesapla.
    #    Aynı görsel kullanıldığı için görsel token sayısı iki taraf arasında tutarlı olur.
    prompt_lengths = []
    for item in batch:
        prompt_msg = build_user_message(item["image_path"])
        prompt_text = processor.apply_chat_template(prompt_msg, tokenize=False, add_generation_prompt=True)
        prompt_image, _ = process_vision_info(prompt_msg)
        prompt_ids = processor(text=[prompt_text], images=prompt_image, padding=False, return_tensors="pt")["input_ids"]
        prompt_lengths.append(prompt_ids.shape[1])

    # 3) labels: prompt kısmı ve padding -100 (yoksay), sadece cevap kısmı gerçek token id
    labels = full_inputs["input_ids"].clone()
    for i, plen in enumerate(prompt_lengths):
        labels[i, :plen] = -100
    labels[full_inputs["attention_mask"] == 0] = -100

    full_inputs["labels"] = labels
    return full_inputs


train_dataset = QwenOCRDataset(train_ids, TRAIN_DIR)
valid_dataset = QwenOCRDataset(valid_ids, VALID_DIR)
print(f"train_dataset: {len(train_dataset)} | valid_dataset: {len(valid_dataset)}")

## 7. 80GB GPU için batch size'ı deneyerek bul

Belgeler A4 sayfa görüntüleri olduğu için tek bir görsel bile binlerce "görsel token" üretebiliyor — bu yüzden VRAM kullanımı sadece model boyutuna değil, **sayfa çözünürlüğüne** de bağlı. Doğru batch size'ı tahmin etmek yerine küçükten başlayıp deneyerek buluyoruz: her aday için gerçek bir forward+backward adımı çalıştırıp OOM (out of memory) alana kadar artırıyoruz.

In [ ]:
def try_batch_size(bs: int) -> bool:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        sample_items = [train_dataset[i] for i in range(bs)]
        batch = collate_fn(sample_items)
        batch = {k: v.to("cuda") if torch.is_tensor(v) else v for k, v in batch.items()}

        model.train()
        outputs = model(**batch)
        outputs.loss.backward()
        model.zero_grad(set_to_none=True)

        peak_gb = torch.cuda.max_memory_allocated() / 1024**3
        print(f"  batch_size={bs}: OK, peak VRAM = {peak_gb:.1f} GB")
        return True
    except torch.cuda.OutOfMemoryError:
        print(f"  batch_size={bs}: OOM (bellek yetmedi)")
        model.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
        return False


candidate_batch_sizes = [2, 4, 6, 8, 12, 16]
last_ok = None
for bs in candidate_batch_sizes:
    if bs > len(train_dataset):
        break
    ok = try_batch_size(bs)
    if ok:
        last_ok = bs
    else:
        break

print(f"\nEn yüksek çalışan batch_size: {last_ok}")
print("Not: Trainer, DataLoader + optimizer state gibi ek bellek kullanacağı için,")
print("güvenlik payı bırakmak amacıyla bulunan değerin biraz altını (örn. 1-2 küçüğünü) kullanmanızı öneririz.")

In [ ]:
# Deneme sonucu: batch_size=2 -> OK (48.4GB / 80GB), batch_size=4 -> OOM.
# Aradaki 3'ü test etmedik ama 2'de hâlâ ~30GB boşluk var; LoRA'nın optimizer state'i
# çok küçük olduğu için (~40M eğitilebilir parametre) bu boşluk Trainer'ın ek yükünü
# rahat karşılar. Bu yüzden last_ok'un formülle küçültülmesi yerine 2'yi doğrudan kullanıyoruz.
PER_DEVICE_BATCH_SIZE = 2

TARGET_EFFECTIVE_BATCH_SIZE = 32  # optimizer adımı başına görmek istediğimiz toplam örnek sayısı
GRADIENT_ACCUMULATION_STEPS = max(1, TARGET_EFFECTIVE_BATCH_SIZE // PER_DEVICE_BATCH_SIZE)

print(f"PER_DEVICE_BATCH_SIZE = {PER_DEVICE_BATCH_SIZE}")
print(f"GRADIENT_ACCUMULATION_STEPS = {GRADIENT_ACCUMULATION_STEPS}")
print(f"Efektif batch size = {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

## 8. Eğitim ayarları ve Trainer

In [ ]:
import inspect

NUM_EPOCHS = 3
LEARNING_RATE = 1e-4

# Colab'ın önceden yüklü transformers sürümü, restart edilmeden pip upgrade sonrası da
# hâlâ bellekte eski TrainingArguments sınıfı olarak kalabiliyor (bazı yeni parametreleri
# tanımıyor, örn. warmup_ratio). Restart gerektirmeden çalışması için, sadece bu sınıfın
# gerçekten kabul ettiği parametreleri filtreleyip gönderiyoruz.
desired_args = dict(
    output_dir=LOCAL_CHECKPOINT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    warmup_steps=0,
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    evaluation_strategy="steps",  # eski sürümler bu ismi kullanıyor, ikisinden biri kabul edilecek
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    report_to="none",
    dataloader_num_workers=2,
    gradient_checkpointing=False,  # zaten model üzerinde elle açtık, Trainer tekrar dokunmasın
)

accepted_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
filtered_args = {k: v for k, v in desired_args.items() if k in accepted_params}
dropped = set(desired_args) - set(filtered_args)
if dropped:
    print("Bu transformers sürümünde desteklenmeyen, atlanan parametreler:", dropped)

training_args = TrainingArguments(**filtered_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=collate_fn,
)

## 9. Eğitimi başlat

`eval_loss` (teacher-forcing loss üzerinden), gerçek CER değil — hızlı takip için. Gerçek CER karşılaştırması eğitim bittikten sonra, holdout `dataset_1000/test/` üzerinde yapılacak (bkz. bölüm 11). Baseline notebook da aynı `test/` yolunu kullanır.

In [ ]:
trainer.train()

## 10. LoRA adaptörünü kaydet (Drive'a kopyala)

In [ ]:
LOCAL_FINAL_ADAPTER_DIR = "/content/lora_final"
model.save_pretrained(LOCAL_FINAL_ADAPTER_DIR)
processor.save_pretrained(LOCAL_FINAL_ADAPTER_DIR)

os.makedirs(LORA_OUTPUT_DIR_DRIVE, exist_ok=True)
shutil.copytree(LOCAL_FINAL_ADAPTER_DIR, LORA_OUTPUT_DIR_DRIVE, dirs_exist_ok=True)
print(f"LoRA adaptörü kaydedildi: {LORA_OUTPUT_DIR_DRIVE}")

## 11. Fine-tune sonrası holdout test setinde (`dataset_1000/test/`) CER ölç

Bu bölüm, fine-tune edilmiş modeli eğitimde **hiç görmediği** `test/` klasöründeki örnekler üzerinde çalıştırıp CER ve Türkçe karakter hata oranını ölçer, grup bazlı özet çıkarır.

In [ ]:
TURKISH_CHARS = set("ıiİIşsğgöoüuçc")


def compute_cer(reference: str, hypothesis: str) -> float:
    if len(reference) == 0:
        return 0.0 if len(hypothesis) == 0 else 1.0
    return jiwer.cer(reference, hypothesis)


def compute_turkish_char_error_rate(reference: str, hypothesis: str) -> dict:
    if len(reference) == 0:
        return {"turkish_char_error_rate": 0.0, "turkish_char_total": 0, "turkish_char_errors": 0}
    output = jiwer.process_characters(reference, hypothesis)
    alignment = output.alignments[0]
    total, errors = 0, 0
    for chunk in alignment:
        if chunk.type == "insert":
            continue
        ref_slice = reference[chunk.ref_start_idx:chunk.ref_end_idx]
        for ref_char in ref_slice:
            if ref_char not in TURKISH_CHARS:
                continue
            total += 1
            if chunk.type != "equal":
                errors += 1
    rate = (errors / total) if total > 0 else 0.0
    return {"turkish_char_error_rate": rate, "turkish_char_total": total, "turkish_char_errors": errors}


def run_ocr_finetuned(image_path: str, max_new_tokens: int = 2048) -> str:
    model.eval()
    processor.tokenizer.padding_side = "left"  # generate() için left-padding
    messages = build_user_message(image_path)
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text_prompt], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    trimmed_ids = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(trimmed_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return output_text.strip()


test_log_path = os.path.join(TEST_DIR, "augmentation_log.json")
test_log = {}
if os.path.isfile(test_log_path):
    with open(test_log_path, "r", encoding="utf-8") as f:
        test_log = json.load(f)

finetuned_results = []
for idx, sid in enumerate(test_ids, start=1):
    with open(os.path.join(TEST_DIR, f"{sid}.txt"), "r", encoding="utf-8") as f:
        gt = f.read()
    pred = run_ocr_finetuned(os.path.join(TEST_DIR, f"{sid}.png"))
    cer = compute_cer(gt, pred)
    tr_stats = compute_turkish_char_error_rate(gt, pred)
    group = test_log.get(sid, {}).get("group", sid.split("_")[0])
    finetuned_results.append({
        "id": sid, "group": group, "cer": cer,
        "turkish_char_error_rate": tr_stats["turkish_char_error_rate"],
        "turkish_char_total": tr_stats["turkish_char_total"],
        "turkish_char_errors": tr_stats["turkish_char_errors"],
        "ground_truth": gt,
        "prediction": pred,
    })
    print(f"[{idx}/{len(test_ids)}] {sid} | CER={cer:.4f}")

print("\nHoldout test tamamlandı.")

In [ ]:
# Grup bazlı özet + en kötü 5 örnek.
# Baseline notebook aynı dataset_1000/test/ holdout'unu kullanır;
# Drive'daki baseline_results.json ile before/after CER kıyası yapılabilir.

groups = sorted(set(r["group"] for r in finetuned_results))
ft_group_summary = {}
for g in groups:
    g_results = [r for r in finetuned_results if r["group"] == g]
    ft_group_summary[g] = {
        "num_samples": len(g_results),
        "avg_cer": sum(r["cer"] for r in g_results) / len(g_results),
        "avg_turkish_char_error_rate": sum(r["turkish_char_error_rate"] for r in g_results) / len(g_results),
    }
ft_overall_cer = sum(r["cer"] for r in finetuned_results) / len(finetuned_results)
ft_overall_tr = sum(r["turkish_char_error_rate"] for r in finetuned_results) / len(finetuned_results)

worst5 = sorted(finetuned_results, key=lambda r: r["cer"], reverse=True)[:5]

print(f"{'Grup':<8}{'Örnek Sayısı':<14}{'Ort. CER':<12}{'Ort. TR Char Hata Oranı':<26}")
print("-" * 60)
for g in groups:
    s = ft_group_summary[g]
    print(f"{g:<8}{s['num_samples']:<14}{s['avg_cer']:<12.4f}{s['avg_turkish_char_error_rate']:<26.4f}")
print("-" * 60)
print(f"{'GENEL':<8}{len(finetuned_results):<14}{ft_overall_cer:<12.4f}{ft_overall_tr:<26.4f}")

print("\nEn kötü 5 örnek:")
for r in worst5:
    print(f"  {r['id']} (grup {r['group']}) - CER={r['cer']:.4f}")

final_output = {
    "genel_ozet": {"num_samples": len(finetuned_results), "avg_cer": ft_overall_cer, "avg_turkish_char_error_rate": ft_overall_tr},
    "grup_bazli_sonuclar": ft_group_summary,
    "en_kotu_5_ornek": [
        {"id": r["id"], "group": r["group"], "cer": r["cer"], "turkish_char_error_rate": r["turkish_char_error_rate"]}
        for r in worst5
    ],
    "ornek_bazli_detaylar": finetuned_results,
}

with open(RESULTS_OUTPUT_PATH_DRIVE, "w", encoding="utf-8") as f:
    json.dump(final_output, f, ensure_ascii=False, indent=2)
print(f"\nKaydedildi: {RESULTS_OUTPUT_PATH_DRIVE}")